In [ ]:
from google.colab import drive

# Google Drive を /content/drive にマウント
drive.mount('/content/drive')

%cd /content/drive/MyDrive/research/proj-spectrum_denoising_1d/spectrum_denoising_1d/examples/
!pip install pytorch_lightning

In this notebook, we train an autoregressive PixelCNN to model noise.
This version (v2) uses **positional-encoding augmented** noise data
with shape `(n_samples, 1, channel, 1+d_model)`.


In [ ]:
import os
import sys

import torch
import numpy as np
import pytorch_lightning as pl
from pytorch_lightning.callbacks import (
    EarlyStopping,
    LearningRateMonitor,
)
from pytorch_lightning.loggers import TensorBoardLogger
import matplotlib.pyplot as plt

sys.path.append("../")
from noise_model.PixelCNN import PixelCNN
from utils.dataloaders import create_nm_loader
from utils.tools import view_receptive_field, autocorrelation

plt.style.use("dark_background")
plt.rc("figure", figsize=[20, 5])

Load noise samples with positional encoding.</br>
Shape: `[Number, 1, Width, 1+d_model]`  
The last axis is `[scalar_noise, pe_1, pe_2, ..., pe_d_model]`.


In [ ]:
at_pbs = np.load("./../../data/for_1d_denoising/pbs_1d_pe.npy")
print("at_pbs.shape", at_pbs.shape)
# Expected: (n_samples, 1, channel, 1+d_model) e.g. (100, 1, 200, 5)

# The mean of noise should be zero.
# For PE data, only subtract mean from the noise channel (index 0).
noise_ch_mean = np.mean(at_pbs[..., 0])
at_pbs[..., 0] = at_pbs[..., 0] - noise_ch_mean

Check temporal autocorrelation of noise (scalar channel only).

In [ ]:
# Use the scalar noise channel (index 0) for autocorrelation
autocorrelation([at_pbs[0, 0, :, 0]], max_lag=50)

Create data loaders and get the mean and standard deviation of the noise samples.</br>

In [ ]:
nm_train_loader, nm_val_loader, noise_mean, noise_std = create_nm_loader(
    at_pbs, batch_size=32, split=0.8
)

Set noise model checkpoint directory.

In [ ]:
nm_checkpoint_path = f"../nm_checkpoint_pe"

Initialise trainer and noise model.

**Key change**: `in_channels` is set to `1 + d_model` (the size of the
last axis of the PE data) so that the CNN treats the PE features as
input channels and convolves only along the *channel* dimension.


In [ ]:
# PE feature dimension = last axis of the data
pe_channels = at_pbs.shape[-1]   # 1 + d_model, e.g. 5
print(f"in_channels (1+d_model) = {pe_channels}")

noise_model = PixelCNN(
    in_channels=pe_channels,
    n_filters=8,
    kernel_size=11,
    n_gaussians=2,
    noise_mean=noise_mean,
    noise_std=noise_std,
    lr=2e-3,
)

use_cuda = torch.cuda.is_available()
trainer = pl.Trainer(
    default_root_dir=nm_checkpoint_path,
    accelerator="gpu" if use_cuda else "cpu",
    devices=1,
    max_epochs=1000,
    logger=TensorBoardLogger(nm_checkpoint_path),
    log_every_n_steps=len(nm_train_loader),
    callbacks=[
        LearningRateMonitor(logging_interval="epoch"),
        EarlyStopping(patience=50, monitor="val/nll"),
    ],
)

View receptive field (now using the 2-D shape `(channel, 1+d_model)`).

In [ ]:
# Pass (channel, 1+d_model) so that view_receptive_field creates
# a 4-D dummy input [1, 1, channel, 1+d_model].
view_receptive_field(noise_model, at_pbs.shape[-2:])

Train and save final parameters.

In [ ]:
trainer.fit(noise_model, nm_train_loader, nm_val_loader)
trainer.save_checkpoint(os.path.join(nm_checkpoint_path, "final_params.ckpt"))